# Dependencies

In [ ]:
%pip install torch transformers>=4.50 peft>=0.15 bitsandbytes>=0.45 accelerate>=1.5 datasets>=3.5 huggingface_hub>=0.30 mlflow>=2.20 sacrebleu>=2.5 pandas>=2.2 numpy>=2.0 python-dotenv>=1.0 sentencepiece

# Code

In [ ]:
import argparse
import json
import os
import random
from contextlib import nullcontext
from pathlib import Path

#EXPERIMENT SETTINGS

# True:
#   - Read Kaggle Secrets
#   - Use MLflow / DagsHub
#   - Upload LoRA adapter to Hugging Face
#
# False:
#   - Do NOT read any Secrets
#   - Do NOT use MLflow / DagsHub
#   - Do NOT upload to Hugging Face
#   - Save the LoRA adapter locally
#
# For reproducibility use FALSE and turn on any GPU environment

USE_TRACKING = False

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import mlflow
import numpy as np
import pandas as pd
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)
from sacrebleu.metrics import BLEU, CHRF
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    Trainer,
    TrainingArguments,
    set_seed,
)
from sklearn.model_selection import train_test_split


#PROMPT

SYSTEM_PROMPT = (
    "You are an expert translator of Akkadian. "
    "Translate the Akkadian transliteration into accurate English. "
    "Return only the English translation."
)


def build_prompt(source: str) -> str:
    return (
        f"{SYSTEM_PROMPT}\n\n"
        f"Akkadian:\n{source}\n\n"
        f"English:\n"
    )


#TOKENIZATION

def tokenize_example(example, tokenizer, max_length):
    prompt = build_prompt(example["source_text"])
    target = example["target_text"]

    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    target_ids = tokenizer(
        target,
        add_special_tokens=False,
        truncation=False,
    )["input_ids"]

    eos = (
        [tokenizer.eos_token_id]
        if tokenizer.eos_token_id is not None
        else []
    )

    available = max_length - len(eos)

    if available <= 0:
        raise ValueError(
            f"max_length={max_length} is too small to fit EOS."
        )

    # Priority:
    # 1. Preserve the prompt.
    # 2. Use remaining tokens for the target.

    if len(prompt_ids) >= available:
        prompt_ids = prompt_ids[:available]
        target_ids = []
    else:
        remaining = available - len(prompt_ids)
        target_ids = target_ids[:remaining]

    input_ids = prompt_ids + target_ids + eos

    labels = (
            [-100] * len(prompt_ids)
            + target_ids
            + eos
    )

    attention_mask = [1] * len(input_ids)

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


#DATASET

class ListDataset(torch.utils.data.Dataset):

    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        return self.records[idx]


#GENERATION

def generate_predictions(
        model,
        tokenizer,
        df,
        max_input_length,
        max_new_tokens,
        batch_size,
):

    model.eval()

    predictions = []

    # Disable generation sampling parameters inherited from
    # the model's generation_config.
    if hasattr(model, "generation_config"):
        model.generation_config.temperature = None
        model.generation_config.top_p = None
        model.generation_config.top_k = None

    for start in range(0, len(df), batch_size):

        batch = df.iloc[start:start + batch_size]

        prompts = [
            build_prompt(source)
            for source in batch["source_text"]
        ]

        enc = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_input_length,
        )

        enc = {
            key: value.to(model.device)
            for key, value in enc.items()
        }

        with torch.no_grad():

            outputs = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for i, sequence in enumerate(outputs):

            prompt_len = int(
                enc["attention_mask"][i].sum().item()
            )

            generated_tokens = sequence[prompt_len:]

            text = tokenizer.decode(
                generated_tokens,
                skip_special_tokens=True,
            ).strip()

            predictions.append(text)

    return predictions


#LENGTH STATISTICS

def print_length_statistics(dataset):

    lengths = [
        len(example["input_ids"])
        for example in dataset
    ]

    if not lengths:
        print("Dataset is empty.")
        return

    print("\n" + "=" * 50)
    print("TOKEN LENGTH STATISTICS")
    print("=" * 50)

    print(f"Max:  {max(lengths)}")
    print(f"Mean: {sum(lengths) / len(lengths):.2f}")

    sorted_lengths = sorted(lengths)

    for percentile in [50, 75, 90, 95, 99]:

        index = min(
            int(len(sorted_lengths) * percentile / 100),
            len(sorted_lengths) - 1,
        )

        value = sorted_lengths[index]

        print(f"P{percentile}: {value}")

    print("=" * 50)


#MAIN

def main():

    parser = argparse.ArgumentParser()

    #Model / data
    parser.add_argument(
        "--model",
        default="Qwen/Qwen2.5-7B-Instruct",
    )

    parser.add_argument(
        "--train",
        default=(
            "/kaggle/input/competitions/"
            "deep-past-initiative-machine-translation/train.csv"
        ),
    )

    parser.add_argument(
        "--val",
        default=(
            "/kaggle/input/competitions/"
            "deep-past-initiative-machine-translation/test.csv"
        ),
    )

    parser.add_argument(
        "--output-dir",
        default="outputs/qwen2.5-7b-lora",
    )

    #Sequence / generation
    parser.add_argument(
        "--max-length",
        type=int,
        default=512,
    )

    parser.add_argument(
        "--max-new-tokens",
        type=int,
        default=256,
    )

    #Training
    parser.add_argument(
        "--epochs",
        type=float,
        default=2.0,
    )

    parser.add_argument(
        "--learning-rate",
        type=float,
        default=2e-4,
    )

    parser.add_argument(
        "--batch-size",
        type=int,
        default=1,
    )

    parser.add_argument(
        "--grad-accum",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--eval-batch-size",
        type=int,
        default=1,
    )

    #LoRA
    parser.add_argument(
        "--lora-r",
        type=int,
        default=16,
    )

    parser.add_argument(
        "--lora-alpha",
        type=int,
        default=32,
    )

    parser.add_argument(
        "--lora-dropout",
        type=float,
        default=0.05,
    )

    #Misc
    parser.add_argument(
        "--seed",
        type=int,
        default=42,
    )

    args, unknown = parser.parse_known_args()

    #ENVIRONMENT
    load_dotenv()

    set_seed(args.seed)
    random.seed(args.seed)
    np.random.seed(args.seed)

    # KAGGLE SECRETS
    # Secrets are read ONLY when USE_TRACKING=True.
    hf_token = None
    hf_repo = None

    if USE_TRACKING:

        print("\nLoading Kaggle Secrets...")

        secrets = UserSecretsClient()

        os.environ["HF_TOKEN"] = secrets.get_secret(
            "HF_TOKEN"
        )

        os.environ["HF_REPO_ID"] = secrets.get_secret(
            "HF_REPO_ID"
        )

        os.environ["DAGSHUB_USERNAME"] = secrets.get_secret(
            "DAGSHUB_USERNAME"
        )

        os.environ["DAGSHUB_TOKEN"] = secrets.get_secret(
            "DAGSHUB_TOKEN"
        )

        os.environ["DAGSHUB_PASSWORD"] = secrets.get_secret(
            "DAGSHUB_PASSWORD"
        )

        os.environ["MLFLOW_TRACKING_URI"] = secrets.get_secret(
            "MLFLOW_TRACKING_URI"
        )

        os.environ["MLFLOW_EXPERIMENT_NAME"] = secrets.get_secret(
            "MLFLOW_EXPERIMENT_NAME"
        )

        #DagsHub credentials for MLflow
        os.environ["MLFLOW_TRACKING_USERNAME"] = (
            os.environ["DAGSHUB_USERNAME"]
        )

        os.environ["MLFLOW_TRACKING_PASSWORD"] = (
            os.environ["DAGSHUB_TOKEN"]
        )

        hf_token = os.getenv("HF_TOKEN")
        hf_repo = os.getenv("HF_REPO_ID")

    #DEVICE
    if not torch.cuda.is_available():
        raise RuntimeError(
            "QLoRA requires a CUDA-capable GPU with bitsandbytes."
        )

    print("\n" + "=" * 60)
    print("DEVICE")
    print("=" * 60)

    print(
        "CUDA available:",
        torch.cuda.is_available(),
    )

    print(
        "CUDA devices:",
        torch.cuda.device_count(),
    )

    print(
        "Device:",
        torch.cuda.get_device_name(0),
    )

    print(
        "Tracking:",
        USE_TRACKING,
    )

    print("=" * 60)

    #AUTH

    if USE_TRACKING:

        if not hf_token:
            raise RuntimeError(
                "HF_TOKEN is missing."
            )

        login(token=hf_token)

    #DATA
    print("\nLoading dataset...")

    train_df = pd.read_csv(args.train)

    print(f"Full dataset size: {len(train_df)}")

    train_df = train_df.rename(
        columns={
            "transliteration": "source_text",
            "translation": "target_text",
        }
    )

    # Keep only valid examples
    train_df["source_text"] = train_df["source_text"].astype(str).str.strip()
    train_df["target_text"] = train_df["target_text"].astype(str).str.strip()

    train_df = train_df[
        (train_df["source_text"] != "")
        & (train_df["target_text"] != "")
        ].reset_index(drop=True)

    #SPLIT

    train_df, val_df = train_test_split(
        train_df,
        test_size=0.1,
        random_state=42,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    print(f"Train size: {len(train_df)}")
    print(f"Val size:   {len(val_df)}")

    #Debugging
    required_columns = [
        "source_text",
        "target_text",
    ]

    for column in required_columns:

        if column not in train_df.columns:
            raise ValueError(
                f"Missing column '{column}' in train dataset."
            )

        if column not in val_df.columns:
            raise ValueError(
                f"Missing column '{column}' in validation dataset."
            )

    #TOKENIZER
    print("\nLoading tokenizer...")

    tokenizer = AutoTokenizer.from_pretrained(
        args.model,
        token=hf_token,
        use_fast=True,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    #QUANTIZATION
    print("\nCreating 4-bit quantization config...")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )

    #MODEL
    print("\nLoading model...")

    model = AutoModelForCausalLM.from_pretrained(
        args.model,
        quantization_config=bnb_config,

        # IMPORTANT:
        # Do NOT use device_map="auto".
        device_map={"": 0},

        dtype=torch.float16,
        token=hf_token,
    )

    model.config.use_cache = False

    # Required for QLoRA.
    model = prepare_model_for_kbit_training(model)

    #LoRA

    peft_config = LoraConfig(
        r=args.lora_r,
        lora_alpha=args.lora_alpha,
        lora_dropout=args.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",

        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
    )

    model = get_peft_model(
        model,
        peft_config,
    )

    print("\nTrainable parameters:")

    model.print_trainable_parameters()

    #TOKENIZE DATA
    print("\nTokenizing train dataset...")

    train_records = [
        tokenize_example(
            row,
            tokenizer,
            args.max_length,
        )
        for row in train_df.to_dict("records")
    ]

    print("Tokenizing validation dataset...")

    val_records = [
        tokenize_example(
            row,
            tokenizer,
            args.max_length,
        )
        for row in val_df.to_dict("records")
    ]

    train_dataset = ListDataset(train_records)
    val_dataset = ListDataset(val_records)

    print_length_statistics(train_dataset)

    #COLLATOR
    collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        label_pad_token_id=-100,
        return_tensors="pt",
    )

    #OUTPUT
    output_dir = Path(args.output_dir)

    output_dir.mkdir(
        parents=True,
        exist_ok=True,
    )

    #TRAINING ARGUMENTS

    training_args = TrainingArguments(
        output_dir=str(output_dir),

        #Training duration
        num_train_epochs=args.epochs,

        #Optimization
        learning_rate=args.learning_rate,

        lr_scheduler_type="cosine",

        warmup_ratio=0.05,

        weight_decay=0.01,

        optim="paged_adamw_8bit",

        #Batch

        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.eval_batch_size,
        gradient_accumulation_steps=args.grad_accum,

        #Precision

        fp16=True,
        bf16=False,

        #Memory

        gradient_checkpointing=True,

        #Evaluation
        eval_strategy="epoch",

        #Checkpoints
        save_strategy="epoch",

        save_total_limit=2,

        # VERY IMPORTANT:
        # After training, Trainer restores the checkpoint
        # with the lowest eval_loss.

        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        #Logging
        logging_steps=10,

        #Do not send Trainer logs to external integrations.
        report_to=[],

        #Reproducibility
        seed=args.seed,
        data_seed=args.seed,
    )

    #MLFLOW SETUP
    if USE_TRACKING:

        mlflow.set_tracking_uri(
            os.getenv("MLFLOW_TRACKING_URI")
        )

        mlflow.set_experiment(
            os.getenv(
                "MLFLOW_EXPERIMENT_NAME",
                "akkadian-qwen2.5-lora",
            )
        )

    #START RUN
    run_context = (
        mlflow.start_run()
        if USE_TRACKING
        else nullcontext(None)
    )

    with run_context as run:

        #MLFLOW PARAMETERS
        if USE_TRACKING:

            mlflow.log_params({
                "base_model": args.model,

                "architecture":
                    "qwen2.5-7b-qlora",

                "learning_rate":
                    args.learning_rate,

                "lr_scheduler":
                    "cosine",

                "warmup_ratio":
                    0.05,

                "weight_decay":
                    0.01,

                "batch_size":
                    args.batch_size,

                "gradient_accumulation":
                    args.grad_accum,

                "max_length":
                    args.max_length,

                "max_new_tokens":
                    args.max_new_tokens,

                "epochs":
                    args.epochs,

                "lora_r":
                    args.lora_r,

                "lora_alpha":
                    args.lora_alpha,

                "lora_dropout":
                    args.lora_dropout,

                "quantization":
                    "4bit-nf4-double-quant",

                "compute_dtype":
                    "float16",

                "gradient_checkpointing":
                    True,

                "optimizer":
                    "paged_adamw_8bit",

                "train_size":
                    len(train_df),

                "val_size":
                    len(val_df),

                "seed":
                    args.seed,
            })

        #TRAINER

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=val_dataset,
            data_collator=collator,
        )

        #TRAIN
        print("\n" + "=" * 60)
        print("START TRAINING")
        print("=" * 60)

        train_result = trainer.train()

        #EVALUATION
        print("\n" + "=" * 60)
        print("FINAL EVALUATION")
        print("=" * 60)

        eval_metrics = trainer.evaluate()

        train_metrics = train_result.metrics

        #LOG LOSSES
        if USE_TRACKING:

            if "train_loss" in train_metrics:

                mlflow.log_metric(
                    "train_loss",
                    float(train_metrics["train_loss"]),
                )

            if "eval_loss" in eval_metrics:

                mlflow.log_metric(
                    "val_loss",
                    float(eval_metrics["eval_loss"]),
                )

        #SAVE BEST MODEL
        print("\nSaving best model...")

        trainer.save_model(
            str(output_dir)
        )

        tokenizer.save_pretrained(
            str(output_dir)
        )


        #GENERATION
        print("\n" + "=" * 60)
        print("GENERATING VALIDATION PREDICTIONS")
        print("=" * 60)

        predictions = generate_predictions(
            model=model,
            tokenizer=tokenizer,
            df=val_df,
            max_input_length=args.max_length,
            max_new_tokens=args.max_new_tokens,
            batch_size=args.eval_batch_size,
        )

        references = (
            val_df["target_text"]
            .astype(str)
            .tolist()
        )

        #BLEU
        bleu = BLEU().corpus_score(
            predictions,
            [references],
        ).score

        #chrF++
        chrfpp = CHRF(
            word_order=2
        ).corpus_score(
            predictions,
            [references],
        ).score

        geom_mean = float(
            np.sqrt(bleu * chrfpp)
        )

        #METRICS
        print("\n" + "=" * 60)
        print("TRANSLATION METRICS")
        print("=" * 60)

        print(
            f"BLEU:  {bleu:.4f}"
        )

        print(
            f"chrF++: {chrfpp:.4f}"
        )

        print(
            f"Geometric mean: {geom_mean}"
        )


        if USE_TRACKING:

            mlflow.log_metric(
                "val_BLEU",
                float(bleu),
            )

            mlflow.log_metric(
                "val_chrf_plus_plus",
                float(chrfpp),
            )

            mlflow.log_metric(
                "geom_mean_score",
                geom_mean,
            )

        #SAVE PREDICTIONS

        pred_df = val_df.copy()

        pred_df["pred_translation"] = predictions

        pred_path = (
                output_dir
                / "preds_val_qwen2.5-7b-lora.csv"
        )

        if "oare_id" in pred_df.columns:

            pred_df.rename(
                columns={
                    "oare_id": "id"
                },
                inplace=True,
            )

        pred_df.to_csv(
            pred_path,
            index=False,
        )

        if USE_TRACKING:

            mlflow.log_artifact(
                str(pred_path),
                artifact_path="predictions",
            )

        #HUGGING FACE
        if USE_TRACKING and hf_repo:

            print("\n" + "=" * 60)
            print("UPLOADING LoRA ADAPTER TO HUGGING FACE")
            print("=" * 60)

            # `model` now contains the best checkpoint.
            model.push_to_hub(
                hf_repo,
                private=True,
                token=hf_token,
                commit_message=(
                    "Upload best Akkadian QLoRA adapter"
                ),
            )

            tokenizer.push_to_hub(
                hf_repo,
                private=True,
                token=hf_token,
                commit_message=(
                    "Upload tokenizer"
                ),
            )

            mlflow.set_tag(
                "hf_repo",
                hf_repo,
            )


        #MLFLOW TAGS
        if USE_TRACKING:

            mlflow.set_tag(
                "model_architecture",
                "qwen2.5-lora",
            )

            mlflow.set_tag(
                "best_model_metric",
                "eval_loss",
            )

        #RUN SUMMARY
        summary = {
            "run_id": (
                run.info.run_id
                if USE_TRACKING and run is not None
                else None
            ),

            "base_model":
                args.model,

            "train_loss":
                train_metrics.get(
                    "train_loss"
                ),

            "val_loss":
                eval_metrics.get(
                    "eval_loss"
                ),

            "val_BLEU":
                float(bleu),

            "val_chrF++":
                float(chrfpp),

            "geom_mean_score":
                geom_mean,

            "epochs":
                args.epochs,

            "learning_rate":
                args.learning_rate,

            "lora_r":
                args.lora_r,

            "lora_alpha":
                args.lora_alpha,

            "max_length":
                args.max_length,

            "train_size":
                len(train_df),

            "val_size":
                len(val_df),

            "hf_repo":
                hf_repo if USE_TRACKING else None,
        }

        if USE_TRACKING:

            mlflow.log_dict(
                summary,
                "run_summary.json",
            )

        #FINAL OUTPUT
        print("\n" + "=" * 60)
        print("TRAINING COMPLETE")
        print("=" * 60)

        print(
            json.dumps(
                summary,
                indent=2,
                ensure_ascii=False,
            )
        )

        print("\nLocal model saved to:")
        print(output_dir)

        if USE_TRACKING:

            print("\nHugging Face repository:")
            print(hf_repo)

        else:

            print("\nTracking disabled.")
            print("No MLflow/DagsHub logging.")
            print("No Hugging Face upload.")


if __name__ == "__main__":
    main()

# Saving my model to Kaggle

In [ ]:
# ============================================================
# Upload trained LoRA adapter → Kaggle Models
# ============================================================
import os
from pathlib import Path
import kagglehub

# ---------- settings ----------
LOCAL_DIR       = Path("outputs/qwen2.5-7b-lora")   # куда сохранил trainer.save_model()
KAGGLE_USERNAME = "userName"
FRAMEWORK       = "pytorch"
MODEL_SLUG      = "akkadika"
VARIATION       = "adapter-v1"

handle = f"{KAGGLE_USERNAME}/{MODEL_SLUG}/{FRAMEWORK}/{VARIATION}"

# ---------- sanity check ----------
assert LOCAL_DIR.exists(), f"Не найдено: {LOCAL_DIR}. Сначала запусти обучение."
print("Files to upload:", sorted(p.name for p in LOCAL_DIR.iterdir()))

# ---------- upload ----------
handle_used = kagglehub.model_upload(
    handle=handle,
    local_model_dir=str(LOCAL_DIR),
    version_notes=(
        "LoRA adapter for Qwen2.5-7B — Akkadika (Akkadian fine-tune). "
        "Requires base Qwen/Qwen2.5-7B-Instruct "
    ),
    license_name="Apache 2.0",
)
print("✅ Uploaded:", handle_used)